# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://images.datacamp.com/image/upload/v1676303379/Marketing/Blog/PySpark_RDD_Cheat_Sheet.pdf) is useful.  As is, the [RDD API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.html).

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

### In this step we prepares the data from raw patent and citation records into a more usable format for further analysis.

### Here we use patentSplit()to extract the patent ID and a specific field from the patent data.

### And we use citationSplit()to extract the citing and cited patent IDs from the citation data.

### This results in two RDDs: 
1. patentData: Contains tuples of patent IDs and their associated data (e.g., state).
2. citationData: Contains tuples of citing and cited patent pairs.

In [6]:
def patentSplit(line):
    line = line.split(',')
    return (line[0], line[5])

def citationSplit(line):  
    line = line.split(',')
    return (line[0], line[1].split('\n')[0])

patentData = rddPatents.map(patentSplit)
citationData = rddCitations.map(citationSplit)

### Here we use take(5) method to retrieve the first 5 elements from an patentData RDD.

In [7]:
patentData.take(5)

[('"PATENT"', '"POSTATE"'),
 ('3070801', '""'),
 ('3070802', '"TX"'),
 ('3070803', '"IL"'),
 ('3070804', '"OH"')]

### Here we use take(5) method to retrieve the first 5 elements from an citationData RDD.

In [8]:
citationData.take(5)

[('"CITING"', '"CITED"'),
 ('3858241', '956203'),
 ('3858241', '1324234'),
 ('3858241', '3398406'),
 ('3858241', '3557384')]

### Here we perform inner join operation between the citationData and patentData RDDs to gather information about each citing patent along with the state information associated with the cited patents.

### In this context, the join will match tuples from both RDDs based on their keys. The resulting RDD will include only the pairs where there is a match in both RDDs.

### Here the cache() is used for caching the result of the join in memory for faster access in subsequent operations. This is useful if you plan to perform multiple actions on firstStep, as it avoids recomputing the join each time.

### And it displays the resulting RDD.

In [9]:
firstStep = citationData.join(patentData).cache()
firstStep.take(5)

[('3858577', ('3456641', '"CA"')),
 ('3858577', ('3467098', '"CA"')),
 ('3858577', ('3471215', '"CA"')),
 ('3858577', ('3769963', '"CA"')),
 ('3859250', ('3344205', '"OK"'))]

### This function is designed to swap the Citing and Cited columns in the firstStep RDD.

### This returns a tuple where the cited patent ID becomes the key, and the tuple (citing, citing_state) becomes the value.

In [10]:
def swapData(x):
    citing, (cited, citing_state) = x
    return (cited, (citing, citing_state))

### This function swaps the data back to its original order, with Citing as the key, and also adds the Cited State to the result.

### The function swaps the key back to the citing patent and creates a tuple with the following structure: (citing_state, cited, cited_state).

In [11]:
def dataSwap(x):
    cited, ((citing, citing_state), cited_state) = x
    return (citing, (citing_state, cited, cited_state))

### Mapping with swapData : This applies the swapData function to each element in firstStep, swapping the citing and cited patents and stores it in dataState.

### Joining dataState with patentData : Here we performs a join between the dataState RDD and patentData based on the cited patent ID and the result is stored in tempStep.

### Mapping with dataSwap : Here we apply the dataSwap function to tempStep, effectively restoring the original structure, but now with both citing and cited states included in the result.

In [12]:
dataState = firstStep.map(swapData)
tempStep = dataState.join(patentData).cache()
stateData = tempStep.map(dataSwap)

### Here we use take(5) method to retrieve the first 5 elements from an dataState.

In [13]:
dataState.take(5)

[('3456641', ('3858577', '"CA"')),
 ('3467098', ('3858577', '"CA"')),
 ('3471215', ('3858577', '"CA"')),
 ('3769963', ('3858577', '"CA"')),
 ('3344205', ('3859250', '"OK"'))]

### Here we use take(5) method to retrieve the first 5 elements from an tempStep.

In [14]:
tempStep.take(5)

[('3537390', (('4018148', '"WI"'), '"MI"')),
 ('3537390', (('3997072', '"KY"'), '"MI"')),
 ('3537390', (('3868903', '"NY"'), '"MI"')),
 ('3537390', (('5123341', '"NM"'), '"MI"')),
 ('3537390', (('5074204', '""'), '"MI"'))]

### Here we use take(5) method to retrieve the first 5 elements from an stateData.

In [15]:
stateData.take(5)

[('4018148', ('"WI"', '3537390', '"MI"')),
 ('3997072', ('"KY"', '3537390', '"MI"')),
 ('3868903', ('"NY"', '3537390', '"MI"')),
 ('5123341', ('"NM"', '3537390', '"MI"')),
 ('5074204', ('""', '3537390', '"MI"'))]

### Here we define filter function where its main purpose  is to filter out records where: 
1. Either citing state or cited state is empty (""). 
2. Citing state and cited state are not equal.

### It returns True if both citing_state and cited_state are not empty and they are equal, meaning it’s a same-state citation. Otherwise, it returns False.

In [16]:
def filters(x):
    citing, (citing_state, cited, cited_state) = x
    return True if (citing_state != '""' and cited_state != '""' and (citing_state == cited_state)) else False

### The counter function prepares the data for counting the occurrences of same-state citations by returning a tuple where the citing patent is the key and the value is 1 for each same-state citation.

### This function returns a tuple where the citing patent ID is the key, and 1 is the value, representing one occurrence of a same-state citation.

In [17]:
def counter(x):
    citing, (citing_state, cited, cited_state) = x
    return (citing, 1)

### stateData.filter(filters) : In this line we apply the filters function to the stateData RDD, keeping only the records where the citing and cited states match (i.e., same-state citations) and removing any records with empty states.

In [18]:
citingCount = stateData.filter(filters)

### In this line of code we apply the counter function to each element in citingCount. Now each citing patent has a corresponding value of 1 for each same-state citation.

### And reduceByKey operation aggregates the count of same-state citations for each citing patent. For each key (citing patent), it sums up the 1s to get the total number of same-state citations.

In [19]:
Citing_Count = citingCount.map(counter).reduceByKey(lambda acc, val: acc + val)

### Here it sorts the Citing_Count RDD by the count of same-state citations in descending order (from highest to lowest).

In [20]:
citingData = Citing_Count.sortBy(lambda x: x[1], ascending = False)

### Here we retrieves the top 10 citing patents with the highest same-state citation counts.

In [21]:
citingData.take(10)

[('5959466', 125),
 ('5983822', 103),
 ('6008204', 100),
 ('5952345', 98),
 ('5998655', 96),
 ('5958954', 96),
 ('5936426', 94),
 ('5913855', 90),
 ('5739256', 90),
 ('5978329', 90)]

### patentKV Function : This function transforms the patent data into a key-value pair, where: the key is the patent number, the value is the rest of the patent data in CSV format (with commas)and empty strings in the patent data are replaced with "null" to handle missing values. The output for this function is a key-value tuple where the key is the patent number, and the value is the rest of the data as a comma-separated string.

### emptyData Function : This function ensures that patents with no corresponding citation count (i.e., patents not found in citingData) are still included, with the count set to 0. The output of this function will be the same tuple which we get in patentKV but with None replaced by 0 if there are no citations.

In [22]:
def patentKV(line):
    line_split = line.split(',')
    # Replace empty strings with None in the patent data
    cleaned_line = ["null" if field == '' else field for field in line_split[1:]]
    return (line_split[0], ",".join(cleaned_line))

def emptyData(x):
    (key, (rest, count)) = x
    # Replace None with 0
    if count is None:
        count = 0
    return (key, (rest, count))

### This applies the patentKV function to the rddPatents, converting each line of patent data into a key-value pair. The key is the patent number, and the value is the rest of the data (as a CSV string).

In [23]:
keyValue = rddPatents.map(patentKV)

### This performs a left outer join between the keyValue RDD (containing the patent data) and the citingData RDD (containing same-state citation counts).

In [24]:
tempStep = keyValue.leftOuterJoin(citingData).cache()

### This line of code replaces any None values in the citation count (from the left join) with 0 using the emptyData function resulting in every patent in the dataset will either have its actual citation count or 0 if there were no citations.

In [25]:
finalData = tempStep.map(emptyData)

### The take(10) retrieves and displays the first 10 records from finalData, which contains the cleaned and joined patent data along with the citation counts.

In [26]:
finalData.take(10)

[('3070931',
  ('1963,1096,null,"US","TX",null,2,null,53,6,68,null,10,null,0,null,null,null,null,null,null,null',
   0)),
 ('3071307',
  ('1963,1096,null,"US","IL",null,1,null,229,6,68,null,11,null,0.562,null,null,null,null,null,null,null',
   0)),
 ('3071490',
  ('1963,1096,null,"US","PA",null,1,null,427,1,12,null,3,null,0.4444,null,null,null,null,null,null,null',
   0)),
 ('3071864',
  ('1963,1103,null,"US","OH",null,2,null,34,1,19,null,1,null,0,null,null,null,null,null,null,null',
   0)),
 ('3072076',
  ('1963,1103,null,"US","NY",null,2,null,105,5,55,null,2,null,0,null,null,null,null,null,null,null',
   0)),
 ('3072547',
  ('1963,1103,null,"US","PA",null,2,null,205,1,19,null,0,null,null,null,null,null,null,null,null,null',
   0)),
 ('3072686',
  ('1963,1103,null,"CH","",null,2,null,552,1,14,null,5,null,0,null,null,null,null,null,null,null',
   0)),
 ('3073053',
  ('1963,1110,null,"US","CA",null,1,null,264,5,51,null,0,null,null,null,null,null,null,null,null,null',
   0)),
 ('3073152'

### Here the sortOrder function extracts the count of same-state citations from each record resulting in the count value, which will be used to sort the data.

### In the second line of the code we sort the finalData RDD using the sortOrder function as the sorting key and the 'ascending=False' flag ensures the data is sorted in descending order, meaning patents with the highest same-state citation counts will come first which is stored in finalOutput.

### Then we retrive top 10 results and put them in top_results and display them.

In [27]:
def sortOrder(x):
    (key, (rest, count)) = x
    return count

finalOutput = finalData.sortBy(lambda x: sortOrder(x), ascending=False)

# Show the top 10 results
top_results = finalOutput.take(10)

In [28]:
for result in top_results:
    print(result)

('5959466', ('1999,14515,1997,"US","CA",5310,2,null,326,4,46,159,0,1,null,0.6186,null,4.8868,0.0455,0.044,null,null', 125))
('5983822', ('1999,14564,1998,"US","TX",569900,2,null,114,5,55,200,0,0.995,null,0.7201,null,12.45,0,0,null,null', 103))
('6008204', ('1999,14606,1998,"US","CA",749584,2,null,514,3,31,121,0,1,null,0.7415,null,5,0.0085,0.0083,null,null', 100))
('5952345', ('1999,14501,1997,"US","CA",749584,2,null,514,3,31,118,0,1,null,0.7442,null,5.1102,0,0,null,null', 98))
('5958954', ('1999,14515,1997,"US","CA",749584,2,null,514,3,31,116,0,1,null,0.7397,null,5.181,0,0,null,null', 96))
('5998655', ('1999,14585,1998,"US","CA",null,1,null,560,1,14,114,0,1,null,0.7387,null,5.1667,null,null,null,null', 96))
('5936426', ('1999,14466,1997,"US","CA",5310,2,null,326,4,46,178,0,1,null,0.58,null,11.2303,0.0765,0.073,null,null', 94))
('5739256', ('1998,13983,1995,"US","CA",70060,2,15,528,1,15,453,0,1,null,0.8232,null,15.1104,0.1124,0.1082,null,null', 90))
('5925042', ('1999,14445,1997,"US","C